# Merged Win Pct Validation

This notebook verifies that `merged_win_pct.json` contains every fighter and that each fighter has matchup entries for every other fighter.

## Section 1: Load Source Files

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

base_dir = Path.cwd()
repo_dir = base_dir if (base_dir / "input").exists() else Path(r"C:\Users\Moudimash99\Documents\GitHub\unmatched_matcher")
fighters_path = repo_dir / "input" / "fighters.json"
merged_win_pct_path = repo_dir / "input" / "merged_win_pct.json"

with fighters_path.open("r", encoding="utf-8") as f:
    fighters_data = json.load(f)

with merged_win_pct_path.open("r", encoding="utf-8") as f:
    merged_win_pct = json.load(f)

print(f"Loaded fighters from: {fighters_path}")
print(f"Loaded merged win pct from: {merged_win_pct_path}")
print(f"Fighter records: {len(fighters_data.get('fighters', []))}")
print(f"Top-level merged fighter entries: {len(merged_win_pct)}")
print("Top-level merged keys sample:", list(merged_win_pct.keys())[:10])

## Section 2: Extract and Normalize Fighter IDs

In [ ]:
def normalize_fighter_id(value):
    if value is None:
        return None
    normalized = str(value).strip().lower()
    normalized = re.sub(r"\s+", "_", normalized)
    normalized = re.sub(r"[^a-z0-9_]+", "", normalized)
    normalized = re.sub(r"_+", "_", normalized).strip("_")
    return normalized or None

fighter_records = fighters_data.get("fighters", [])
raw_fighter_ids = [fighter.get("id") for fighter in fighter_records]
normalized_fighter_ids = [normalize_fighter_id(fighter_id) for fighter_id in raw_fighter_ids]
canonical_fighter_ids = [fighter_id for fighter_id in normalized_fighter_ids if fighter_id is not None]
canonical_fighter_set = set(canonical_fighter_ids)

normalized_duplicates = sorted({fighter_id for fighter_id in canonical_fighter_ids if canonical_fighter_ids.count(fighter_id) > 1})

print(f"Canonical fighter ids: {len(canonical_fighter_set)}")
print(f"Duplicate normalized ids: {len(normalized_duplicates)}")
if normalized_duplicates:
    print("Duplicate ids:", normalized_duplicates)

canonical_fighter_df = pd.DataFrame({"fighter_id": canonical_fighter_ids})
display(canonical_fighter_df.head(20))

## Section 3: Check Fighter Coverage in the Merged Win Pct File

In [ ]:
merged_top_level_ids = {normalize_fighter_id(fighter_id) for fighter_id in merged_win_pct.keys()}
merged_top_level_ids = {fighter_id for fighter_id in merged_top_level_ids if fighter_id is not None}

missing_top_level_fighters = sorted(canonical_fighter_set - merged_top_level_ids)
unexpected_top_level_fighters = sorted(merged_top_level_ids - canonical_fighter_set)

print(f"Missing top-level fighters: {len(missing_top_level_fighters)}")
print(f"Unexpected top-level fighters: {len(unexpected_top_level_fighters)}")

coverage_df = pd.DataFrame({
    "fighter_id": sorted(canonical_fighter_set),
    "present_in_merged": [fighter_id in merged_top_level_ids for fighter_id in sorted(canonical_fighter_set)],
})
display(coverage_df)

if missing_top_level_fighters:
    print("Missing top-level fighter ids:", missing_top_level_fighters)
if unexpected_top_level_fighters:
    print("Unexpected top-level fighter ids:", unexpected_top_level_fighters)

## Section 4: Verify Opponent Coverage for Every Fighter

In [ ]:
all_expected_pairs = []
missing_pair_entries = []

for fighter_id in sorted(canonical_fighter_set):
    opponent_map = merged_win_pct.get(fighter_id)
    if not isinstance(opponent_map, dict):
        opponent_map = {}

    merged_opponent_ids = {normalize_fighter_id(opponent_id) for opponent_id in opponent_map.keys()}
    merged_opponent_ids = {opponent_id for opponent_id in merged_opponent_ids if opponent_id is not None}

    expected_opponents = canonical_fighter_set
    all_expected_pairs.append({
        "fighter_id": fighter_id,
        "expected_opponents": len(expected_opponents),
        "present_opponents": len(merged_opponent_ids & expected_opponents),
        "missing_opponents": len(expected_opponents - merged_opponent_ids),
    })

    for opponent_id in sorted(expected_opponents):
        if opponent_id not in merged_opponent_ids:
            missing_pair_entries.append({
                "fighter_id": fighter_id,
                "opponent_id": opponent_id,
                "issue": "missing opponent entry",
            })

opponent_coverage_df = pd.DataFrame(all_expected_pairs)
display(opponent_coverage_df.head(20))

print(f"Total missing pair entries: {len(missing_pair_entries)}")

## Section 5: List Missing Fighters and Missing Pair Entries

In [ ]:
missing_fighters_df = pd.DataFrame({"fighter_id": missing_top_level_fighters})
missing_pairs_df = pd.DataFrame(missing_pair_entries)

print("Missing fighters table:")
display(missing_fighters_df)

print("Missing pair entries table:")
display(missing_pairs_df.head(200))

print(f"Missing fighters rows: {len(missing_fighters_df)}")
print(f"Missing pair rows: {len(missing_pairs_df)}")

## Section 6: Summarize Validation Results

In [ ]:
total_fighters = len(canonical_fighter_set)
matched_fighters = total_fighters - len(missing_top_level_fighters)
expected_pairings = total_fighters * total_fighters
missing_pairings = len(missing_pair_entries)
matched_pairings = expected_pairings - missing_pairings
validation_passed = (
    len(missing_top_level_fighters) == 0
    and len(unexpected_top_level_fighters) == 0
    and missing_pairings == 0
)

summary_df = pd.DataFrame([
    {"metric": "total_fighters", "value": total_fighters},
    {"metric": "matched_fighters", "value": matched_fighters},
    {"metric": "missing_fighters", "value": len(missing_top_level_fighters)},
    {"metric": "unexpected_top_level_fighters", "value": len(unexpected_top_level_fighters)},
    {"metric": "expected_pairings", "value": expected_pairings},
    {"metric": "matched_pairings", "value": matched_pairings},
    {"metric": "missing_pairings", "value": missing_pairings},
    {"metric": "validation_passed", "value": validation_passed},
])

display(summary_df)
print("Validation passed:" if validation_passed else "Validation failed:", validation_passed)